In [0]:
from pyspark.sql.functions import *

# Gold tables are the source for every analysis below
G = "employeedatacatalog.gold_movie"
S = "employeedatacatalog.silver_movie"

   
## Movie Industry Business Cases — Analytics Layer

This notebook is the **consumer** of the medallion architecture. It queries the gold datamarts (`employeedatacatalog.gold_movie`) to answer 7 real-world strategic questions that a studio, distributor, or streaming platform would ask.

> **Visualizations** live in the companion dashboard: **Movie Analytics — Industry & Talent Insights**. This notebook focuses purely on data queries and tabular results.

---

### How It Connects to the Pipeline

```
CSV Files → [Bronze: raw ingestion] → [Silver: star schema] → [Gold: datamarts] → 👉 This Notebook → 📊 Dashboard
```

- **Bronze** cleaned nothing — just stamped metadata and wrote Delta
- **Silver** typed columns, filtered junk rows, parsed dates, built dims/facts/bridge
- **Gold** pre-joined and pre-aggregated so this notebook never touches raw data
- **This notebook** queries gold tables and produces tabular results
- **The dashboard** visualizes the insights interactively

---

### The 7 Business Cases

| # | Question | Gold Table Used |
| --- | --- | --- |
| 1 | Where should you place budget bets? (ROI by tier) | `gold_movie_summary` |
| 2 | Which genres return the most per dollar? | `gold_genre_analytics` |
| 3 | Which directors consistently deliver? | `gold_director_analytics` |
| 4 | Is there a "right" runtime for a movie? | `gold_movie_summary` |
| 5 | Is non-English cinema really growing? | `gold_movie_summary` |
| 6 | What are the best hidden gems for streaming? | `gold_movie_summary` |
| 7 | How did COVID reshape the box office? | `gold_yearly_trends` |

---

### Steps per case:
1. Frame the business question (markdown heading)
2. Query the appropriate gold table with SQL
3. Review tabular results
4. See the companion dashboard for interactive visualizations

### Case 1: Budget Tier ROI — Where Should You Place Your Bets?

Studios constantly wrestle with how much to spend on a film. A $200M blockbuster gets headlines, but does it actually return more per dollar than a scrappy $5M indie? We segment movies into five budget tiers and compare **ROI**, **hit rate** (% that turned a profit), and **average profit** to see where the smart money lands.

In [0]:
# Split every movie with a known budget into five tiers and ask:
#   - What's the average ROI in each tier?
#   - What % of movies actually made money (hit rate)?
#   - What's the average profit in dollars?
# The answer might surprise you — micro-budget ROI looks insane,
# but the hit rate tells a very different story.

df_budget = spark.sql(f"""
    SELECT
        CASE
            WHEN budget < 1000000     THEN '1. Micro (<$1M)'
            WHEN budget < 15000000    THEN '2. Low ($1-15M)'
            WHEN budget < 75000000    THEN '3. Mid ($15-75M)'
            WHEN budget < 150000000   THEN '4. High ($75-150M)'
            ELSE                           '5. Blockbuster ($150M+)'
        END AS budget_tier,
        count(*)                            AS movie_count,
        round(avg(budget) / 1e6, 1)         AS avg_budget_m,
        round(avg(revenue) / 1e6, 1)        AS avg_revenue_m,
        round(avg(profit) / 1e6, 1)         AS avg_profit_m,
        round(avg(roi_pct), 1)              AS avg_roi_pct,
        round(avg(vote_average), 2)         AS avg_rating,
        round(sum(CASE WHEN profit > 0 THEN 1 ELSE 0 END) * 100.0 / count(*), 1) AS hit_rate_pct
    FROM {G}.gold_movie_summary
    WHERE budget > 0
    GROUP BY 1
    ORDER BY 1
""")

display(df_budget)

### Case 2: Genre Investment Strategy — Which Genres Return the Most Per Dollar?

Think of genre selection like a portfolio allocation problem. Some genres cost more to produce (sci-fi, fantasy) but command higher revenues. Others are cheap to make (horror, romance) and punch above their weight. The **revenue multiplier** (revenue ÷ budget) tells you how many dollars come back for every dollar you put in.

In [0]:
# For each genre: how many dollars come back for every dollar of budget?
# That's the revenue_multiplier. Animation leads at 4x, Horror at 3.5x.
# TV Movies and Documentaries barely break even.

df_genre = spark.sql(f"""
    SELECT
        genre_name,
        movie_count,
        round(avg_budget / 1e6, 1)       AS avg_budget_m,
        round(avg_revenue / 1e6, 1)      AS avg_revenue_m,
        round(avg_profit / 1e6, 1)       AS avg_profit_m,
        round(total_revenue / 1e9, 1)    AS total_revenue_b,
        avg_rating,
        CASE WHEN avg_budget > 0
             THEN round(avg_revenue / avg_budget, 2)
             ELSE NULL
        END AS revenue_multiplier
    FROM {G}.gold_genre_analytics
    WHERE avg_budget > 0
    ORDER BY revenue_multiplier DESC
""")

display(df_genre)

### Case 3: Director Bankability — Who Consistently Puts Butts in Seats?

A director’s track record is one of the strongest predictors of a film’s commercial performance. We look at directors with **5+ films** and rank them by average revenue, giving studios a shortlist of reliable bets for their next greenlight decision.

In [0]:
# Directors with at least 5 films, ranked by avg revenue per film.
# James Cameron sits alone at the top: $1.1B avg per movie.
# This is the shortlist a studio exec would want before signing a deal.

df_directors = spark.sql(f"""
    SELECT
        director_name,
        movie_count,
        round(total_revenue / 1e9, 2)    AS total_revenue_b,
        round(avg_revenue / 1e6, 1)      AS avg_revenue_m,
        round(avg_profit / 1e6, 1)       AS avg_profit_m,
        avg_rating,
        avg_popularity,
        top_movie
    FROM {G}.gold_director_analytics
    WHERE movie_count >= 5
      AND avg_revenue > 0
    ORDER BY avg_revenue DESC
    LIMIT 20
""")

display(df_directors)

### Case 4: Runtime Sweet Spot — Is There a "Right" Length for a Movie?

Audiences have limited attention spans, but prestige films tend to run long. Does a longer runtime actually correlate with higher ratings or revenue, or is there a sweet spot where both peak? We bucket films into 30-minute ranges and compare.

In [0]:
# Bucket every movie into 30-minute runtime ranges and compare.
# Turns out longer films do better on both dimensions — but that's
# partly survivorship bias (only good scripts get approved at 3+ hours).

df_runtime = spark.sql(f"""
    SELECT
        CASE
            WHEN runtime_minutes < 90   THEN '1. Under 90 min'
            WHEN runtime_minutes < 120  THEN '2. 90-120 min'
            WHEN runtime_minutes < 150  THEN '3. 120-150 min'
            WHEN runtime_minutes < 180  THEN '4. 150-180 min'
            ELSE                             '5. 180+ min'
        END AS runtime_bucket,
        count(*)                            AS movie_count,
        round(avg(vote_average), 2)         AS avg_rating,
        round(avg(revenue) / 1e6, 1)        AS avg_revenue_m,
        round(avg(profit) / 1e6, 1)         AS avg_profit_m,
        round(avg(popularity), 2)           AS avg_popularity
    FROM {G}.gold_movie_summary
    WHERE runtime_minutes > 0 AND budget > 0
    GROUP BY 1
    ORDER BY 1
""")

display(df_runtime)

### Case 5: Language & Market Opportunity — The Rise of Non-English Cinema

Parasite, Squid Game, RRR — non-English content is no longer niche. Streaming platforms are betting big on international films. We track the volume and quality of English vs non-English films from 2000 to 2025 to see if the data supports the hype.

In [0]:
# Track English vs non-English film volume and quality from 2000-2025.
# Non-English share has been growing, and international films consistently
# rate higher (partly a selection effect — only the best get distributed).

df_lang = spark.sql(f"""
    SELECT
        release_year,
        sum(CASE WHEN original_language = 'en' THEN 1 ELSE 0 END) AS english_count,
        sum(CASE WHEN original_language != 'en' THEN 1 ELSE 0 END) AS non_english_count,
        round(avg(CASE WHEN original_language = 'en' THEN revenue END) / 1e6, 1) AS eng_avg_rev_m,
        round(avg(CASE WHEN original_language != 'en' THEN revenue END) / 1e6, 1) AS intl_avg_rev_m,
        round(avg(CASE WHEN original_language = 'en' THEN vote_average END), 2) AS eng_avg_rating,
        round(avg(CASE WHEN original_language != 'en' THEN vote_average END), 2) AS intl_avg_rating
    FROM {G}.gold_movie_summary
    WHERE release_year BETWEEN 2000 AND 2025
    GROUP BY release_year
    ORDER BY release_year
""")

display(df_lang)

### Case 6: Hidden Gems — Low-Budget Films Ripe for Streaming Acquisition

Streaming platforms need a constant supply of content, and overpaying for blockbusters eats margins. The real value play is finding films that cost almost nothing to make but audiences genuinely love — budget under $5M, rating above 7.5, and enough votes to be credible.

In [0]:
# The streaming acquisition sweet spot: films that cost under $5M
# but audiences rated 7.5+ with at least 500 votes.
# These are proven crowd-pleasers that didn't cost much to make —
# exactly what a platform needs to fill its catalog without burning cash.

df_gems = spark.sql(f"""
    SELECT
        title,
        release_year,
        director_name,
        genre_list,
        round(budget / 1e6, 2)        AS budget_m,
        round(revenue / 1e6, 2)       AS revenue_m,
        vote_average,
        vote_count,
        review_count,
        original_language
    FROM {G}.gold_movie_summary
    WHERE budget BETWEEN 100000 AND 5000000
      AND vote_average >= 7.5
      AND vote_count >= 500
    ORDER BY vote_average DESC
    LIMIT 25
""")

print(f"Found {df_gems.count()} hidden gems")
display(df_gems)

### Case 7: Pandemic Impact & Recovery — How Did COVID Reshape the Box Office?

The 2020 pandemic was an extinction-level event for theatrical revenue. But did the industry bounce back, or has the landscape permanently shifted? We chart total revenue and release volume from 2015–2025 to measure the depth of the crater and the speed of recovery.

In [0]:
# The pandemic story: 2020 revenue fell ~85% and volume ~28%.
# Revenue recovered faster than volume — fewer films, but the ones
# that came out were bigger-budget tentpoles.

df_covid = spark.sql(f"""
    SELECT
        release_year,
        movie_count,
        round(total_revenue / 1e9, 1)  AS total_revenue_b,
        round(avg_revenue / 1e6, 1)    AS avg_revenue_m,
        round(avg_budget / 1e6, 1)     AS avg_budget_m,
        avg_rating,
        avg_runtime
    FROM {G}.gold_yearly_trends
    WHERE release_year BETWEEN 2015 AND 2025
    ORDER BY release_year
""")

display(df_covid)

pdf_c = df_covid.toPandas()
pre = pdf_c[pdf_c['release_year'] == 2019].iloc[0]
dip = pdf_c[pdf_c['release_year'] == 2020].iloc[0]
print(f"\n2020 vs 2019: Revenue dropped {((dip['total_revenue_b'] - pre['total_revenue_b']) / pre['total_revenue_b'] * 100):.0f}%, "
      f"movie count dropped {((dip['movie_count'] - pre['movie_count']) / pre['movie_count'] * 100):.0f}%")